### Step 1: Importing Libraries and setting up API Call

In [1]:
import os
from openai import OpenAI
from IPython.display import display, Markdown
from dotenv import load_dotenv
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ("OPEN API KEY IS MISSING")
else:
    print(OPENAI_API_KEY[:8])

sk-proj-


### Step 2: Creating the Pushover Notif

In [2]:
#2a. set up on browser
#2b. set up on app
#2c. on browser copy user key and save in .env as PUSHOVER_USER = XXXXXXX
#2d. on browser copy API Token and save in .env as PUSHOVER_TOKEN = YYYYYY
#2e. Reload load_dotenv() because you are now wanting to bring in .env (MAKE SURE TO SAVE.ENV)
load_dotenv()

True

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = 'https://api.pushover.net/1/messages.json'

In [4]:
print(pushover_user)
print(pushover_token)

ufvzc85cjkm1asuu7vnd9bf3bwkqgm
akz1oijso1mzepu7axxqthtd8nxov9


### Sending the notification to myself

In [5]:
import requests

def send_notifications(message:str):
    payload = {
        "user" : pushover_user,
        "token" : pushover_token,
        "message" : message
    }

    response = requests.post(url=pushover_url, data=payload)

    # That lets you see whether Pushover said “success” or “something broke.”
    print(response.status_code)
    print(response.text)
    

In [6]:
send_notifications("so cool am i right")

200
{"status":1,"request":"df49c277-aa06-432b-9329-cfc447063ad9"}


### Step 3: Describe Pushover as an LLM tool

In [7]:
send_notification_function = {
    "name" : "send_notifications",
    "description" : "Sends push notifications to the user's phone using Pushover. Use this tool to alert the user about the notification that is important.",
    "parameters" : {
        "type" :"object",
        "properties" : {
            "message": {
                "type" : "string",
                "description" : " The notification message that is sent to the user's device"
            }
        },
        "required" : ['message']
    }
}

### Step 2b & 4b: Create new function, describe it, add it to list of tools

In [8]:
import random 

#Creating the function:
def dice_roll():
    result = random.randint(1,6)
    return result

#describing the function for the LLM:

roll_dice_function = {
'name' : "dice_roll",
'description': 'Receive a random number from the range 1 to 6 everytime you call this function',
'parameters': {
    "type" : "object",
    "properties" : {},
        "required" : []

    }
}
tools = []
#adding to the tools
tools = [{"type" : "function", "function" : roll_dice_function}]

### Step 4: Add Pushover to the list of tools for the LLM

### Adding this to our list of tools

In [9]:
tools.append({"type" : "function", "function":send_notification_function})
tools

[{'type': 'function',
  'function': {'name': 'dice_roll',
   'description': 'Receive a random number from the range 1 to 6 everytime you call this function',
   'parameters': {'type': 'object', 'properties': {}, 'required': []}}},
 {'type': 'function',
  'function': {'name': 'send_notifications',
   'description': "Sends push notifications to the user's phone using Pushover. Use this tool to alert the user about the notification that is important.",
   'parameters': {'type': 'object',
    'properties': {'message': {'type': 'string',
      'description': " The notification message that is sent to the user's device"}},
    'required': ['message']}}}]

In [10]:
# tool_message = response.choices[0].message
# print(tool_message)

In [11]:
def handle_tool_call(tool_calls:list):
        

        tool_call_results = []

        for tool_call in tool_calls:

            function_name = tool_call.function.name

            #  --- for debugging
            # print(f"the function name is {function_name}")
                
            # tool_call = tools_calls[0] #this is because we only have one tool thus far
            
            args = json.loads(tool_call.function.arguments)

            if function_name == 'send_notifications':
                send_notifications(args['message']) #sent to pushover

                content = f"Sent Notification: {args['message']}"

            elif function_name == "dice_roll":

                content =  f' Rolled: {dice_roll()}'

            # elif function_name == "insert_function3_name":

            #     content =insert_function3_name (args['message']})

            else:
                 content = f"Unknown function: {function_name}"

            #package into a dictionary for the llm to see what we got out from the tool call
            tool_call_result = {
            'role' : 'tool',
            'content' : content ,
                'tool_call_id' : tool_call.id   
            }

            print(f"this is what the tool_call_result from handle tool_call looks like: {tool_call_result}")

            #appending each tool_call_result to the tool_call_results list of dictionaries
            tool_call_results.append(tool_call_result)

        return tool_call_results

In [12]:
# Calling the LLM to use the tool calling above
messages= [ 
        {"role" : "user", "content": "Please do two things: 1. I would like to roll a dice 2 times, and 2. Send me a notification with the highest of the rolls"}
    ]

client = OpenAI(api_key=OPENAI_API_KEY)

#give the tool to the model
response = client.chat.completions.create(
    model = 'gpt-4.1-mini',
    messages= messages,
    tools=tools,
    tool_choice= 'auto'
)

#messages = [user request]
# response → OpenAI says "roll dice twice"
# message = "roll dice, roll dice"

message = response.choices[0].message
print(f'initial message that is sent to the LLM before the tool call request {messages}') 

while message.tool_calls :
    from pprint import pprint
    pprint(message.tool_calls)

    tool_call_result = handle_tool_call(message.tool_calls) #so llm knows what came from tool call result and can respond back to the user what it sent

    #append the origianl message of the first tool call initiation
    messages.append(message)
    # print('\n')
    # print(f'initial message that is sent to the LLM should have tool call request {messages}') #assistant reply - should have tool call initiated

    #extend is needed here instead of append this is because if i do append it would be like taking a list [1,2,3] and adding list [4,5] and making it [1,2,3,[4,5]]
    #but extend would extend the list instead of embedding it with its own structure so [1,2,3,4,5]
    #adds that it rolled a 2,5
    messages.extend(tool_call_result) 
    # print('\n')
    # print(f'has messages about the packaged pushover result {messages}') #assistant reply - should have tool call initiated
    # print('\n')

    response = client.chat.completions.create(
        model = 'gpt-4.1-mini',
        messages=messages,
        tools = tools
    )

    message = response.choices[0].message

print(f'final message by llm post everything: {message.content}')
      



initial message that is sent to the LLM before the tool call request [{'role': 'user', 'content': 'Please do two things: 1. I would like to roll a dice 2 times, and 2. Send me a notification with the highest of the rolls'}]
[ChatCompletionMessageFunctionToolCall(id='call_f4VlhAy4yffkIVagzOSyyAVt', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_x8Vo5EuvcafSxukWAo1hONMY', function=Function(arguments='{}', name='dice_roll'), type='function')]
this is what the tool_call_result from handle tool_call looks like: {'role': 'tool', 'content': ' Rolled: 2', 'tool_call_id': 'call_f4VlhAy4yffkIVagzOSyyAVt'}
this is what the tool_call_result from handle tool_call looks like: {'role': 'tool', 'content': ' Rolled: 3', 'tool_call_id': 'call_x8Vo5EuvcafSxukWAo1hONMY'}
[ChatCompletionMessageFunctionToolCall(id='call_ji5QjWXX1IMovBmMqqQCewya', function=Function(arguments='{"message":"The highest roll from the two dice rolls is 3."}',

In [13]:
messages = [ 
    {"role": "user", "content": "Please do two things: 1. I would like to roll a dice 2 times, and 2. Send me a notification with the highest of the rolls"}
]

client = OpenAI(api_key=OPENAI_API_KEY)

response = client.chat.completions.create(
    model='gpt-4.1-mini',
    messages=messages,
    tools=tools,
    tool_choice='auto'
)

message = response.choices[0].message

print("=" * 60)
print("BEFORE LOOP")
print("=" * 60)
print(f"User request in messages: {messages}")
print(f"OpenAI first response (what it wants to do): {message}")
print(f"Tool calls requested: {message.tool_calls}")

loop_count = 0

while message.tool_calls:
    loop_count += 1
    print("\n" + "=" * 60)
    print(f"LOOP ITERATION {loop_count}")
    print("=" * 60)
    print(f"message.tool_calls at start of this iteration: {message.tool_calls}")

    tool_call_result = handle_tool_call(message.tool_calls)

    print(f"\nAfter handle_tool_call, results are: {tool_call_result}")

    messages.append(message)
    print(f"\nAfter messages.append(message), messages is now: {messages}")

    messages.extend(tool_call_result)
    print(f"\nAfter messages.extend(tool_call_result), messages is now: {messages}")

    response = client.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        tools=tools
    )

    #this message is when openai decides on send notifications after seeing the 2 roll dice requests and it decides the message of the assistant to be send notifications and then
    #when while loop starts again this assitant reply to message will be appeneded to the big message with the two tool call requests
    message = response.choices[0].message
    print(f"\nOpenAI response after seeing full history: {message}")
    print(f"Does OpenAI want another tool call? {bool(message.tool_calls)}")

print("\n" + "=" * 60)
print("LOOP ENDED")
print("=" * 60)
print(f"Total loop iterations: {loop_count}")
print(f"Final message from OpenAI: {message.content}")

BEFORE LOOP
User request in messages: [{'role': 'user', 'content': 'Please do two things: 1. I would like to roll a dice 2 times, and 2. Send me a notification with the highest of the rolls'}]
OpenAI first response (what it wants to do): ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_zYccwnpK71YiSifZuHDYO4ML', function=Function(arguments='{}', name='dice_roll'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_99saxkOBgNXwAhom9XNMyHPJ', function=Function(arguments='{}', name='dice_roll'), type='function')])
Tool calls requested: [ChatCompletionMessageFunctionToolCall(id='call_zYccwnpK71YiSifZuHDYO4ML', function=Function(arguments='{}', name='dice_roll'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_99saxkOBgNXwAhom9XNMyHPJ', function=Function(arguments='{}', name='dice_roll'), type='function')]

LOOP ITERATION 1
message.tool